# 🧱 Lab: Multi-Step Reasoning Engine

**Module 3: Prompt Engineering** | **Duration: ~1.5 hours** | **Type: Wall Lab**

## Concepts: chain-of-thought, self-consistency, prompt-chaining

## Learning Objectives
1. Implement Chain-of-Thought prompting for complex reasoning
2. Use Self-Consistency for robust answers
3. Build prompt chains for multi-step problems

In [ ]:
from openai import OpenAI
from dotenv import load_dotenv
from collections import Counter
import re
from IPython.display import Markdown, display

def md(text):
    """Display text as rendered markdown."""
    display(Markdown(text))

load_dotenv()
client = OpenAI()

# Test problems
# Answer is 11
MATH_PROBLEM = '''Roger has 5 tennis balls. He buys 2 more cans of tennis balls.
Each can has 3 tennis balls. How many tennis balls does he have now?'''

# Harder problem that models often get wrong (correct answer: 5 cents)
HARD_MATH_PROBLEM = '''A bat and a ball cost $1.10 together. 
The bat costs $1.00 more than the ball. 
How much does the ball cost in cents?'''

# Another tricky problem (correct answer: 60 students)
TRICKY_PROBLEM = '''In a school, 1/2 of the students study French, 1/3 study Spanish, 
and 1/6 study both languages. If 20 students study neither language, 
how many students are in the school?'''

LOGIC_PROBLEM = '''If all roses are flowers and some flowers fade quickly,
can we conclude that some roses fade quickly?'''

print('Problems loaded!')
print(f'Easy problem: Tennis balls')
print(f'Hard problem: Bat and ball (tricky!)')  
print(f'Tricky problem: Language students')

Problems loaded!
Easy problem: Tennis balls
Hard problem: Bat and ball (tricky!)
Tricky problem: Language students


## 2. Chain-of-Thought Prompting (~20 min)

CoT encourages the model to break down problems into steps.

In [18]:
def solve_with_cot(problem):
    '''Solve problem using Chain-of-Thought prompting.'''
    prompt = f'''Solve this problem step by step.

Problem: {problem}

Let me work through this step by step:'''
    
    response = client.chat.completions.create(
        model='gpt-3.5-turbo',
        messages=[{'role': 'user', 'content': prompt}])
    return response.choices[0].message.content

def solve_without_cot(problem):
    '''Solve problem directly without Chain-of-Thought prompting.'''
    prompt = f'''Answer this question. Just report the final answer.:

{problem}'''
    
    response = client.chat.completions.create(
        model='gpt-3.5-turbo',
        messages=[{'role': 'user', 'content': prompt}])
    return response.choices[0].message.content

md("\n---\n\n> **Notice:** Without CoT, the model may give a correct answer but doesn't show its reasoning. For more complex problems, this approach is more likely to make errors.")


---

> **Notice:** Without CoT, the model may give a correct answer but doesn't show its reasoning. For more complex problems, this approach is more likely to make errors.

In [15]:
md(f"\n**Problem:** {MATH_PROBLEM}")
md("> **Correct answer: 11 tennis balls**")

# Solve math problem with CoT
md("### 🧮 Math Problem with CoT")
md(solve_with_cot(MATH_PROBLEM))

# Solve math problem without CoT
md("### 🧮 Math Problem without CoT")
md(solve_without_cot(MATH_PROBLEM))


**Problem:** Roger has 5 tennis balls. He buys 2 more cans of tennis balls.
Each can has 3 tennis balls. How many tennis balls does he have now?

> **Correct answer: 11 tennis balls**

### 🧮 Math Problem with CoT



1. Roger starts with 5 tennis balls.
2. He buys 2 cans of tennis balls, each containing 3 tennis balls.
3. Number of new tennis balls = 2 cans * 3 balls/can = 6 balls.
4. Total number of tennis balls Roger has now = 5 (initial balls) + 6 (new balls) = 11 tennis balls.

Therefore, Roger now has a total of 11 tennis balls.

### 🧮 Math Problem without CoT

The final answer is 11 tennis balls.

In [19]:
md(f"\n**Problem:** {TRICKY_PROBLEM}")
md("> **Correct answer: 60 students**")

# Solve tricky problem with CoT
md("### ✅ Tricky Problem with CoT")
md(solve_with_cot(TRICKY_PROBLEM))

# Solve tricky problem without CoT
md("### ❌ Tricky Problem WITHOUT CoT (Direct)")
md(solve_without_cot(TRICKY_PROBLEM))


**Problem:** In a school, 1/2 of the students study French, 1/3 study Spanish, 
and 1/6 study both languages. If 20 students study neither language, 
how many students are in the school?

> **Correct answer: 60 students**

### ✅ Tricky Problem with CoT



1. Let's denote the total number of students in the school as x.

2. From the information given, we can set up the following equation:
(1/2)x + (1/3)x - (1/6)x + 20 = x

3. Simplify the equation by combining like terms:
(3/6)x + (2/6)x - (1/6)x + 20 = x
(4/6)x + 20 = x
(2/3)x + 20 = x

4. Solve for x:
(2/3)x = x - 20
2x = 3x - 60
2x - 3x = -60
-x = -60
x = 60

5. Therefore, there are 60 students in the school.

### ❌ Tricky Problem WITHOUT CoT (Direct)

The final answer is 120 students.

In [20]:
# Now let's try the HARD problem - this is where CoT really matters!
md("---\n## 🎯 Hard Problem: Bat and Ball (Classic Cognitive Trap)")
md("> **Correct answer: 5 cents** (most people intuitively say 10 cents)")

md("\n### ❌ WITHOUT Chain-of-Thought:")
md(solve_without_cot(HARD_MATH_PROBLEM))

md("\n### ✅ WITH Chain-of-Thought:")
md(solve_with_cot(HARD_MATH_PROBLEM))

---
## 🎯 Hard Problem: Bat and Ball (Classic Cognitive Trap)

> **Correct answer: 5 cents** (most people intuitively say 10 cents)


### ❌ WITHOUT Chain-of-Thought:

5 cents


### ✅ WITH Chain-of-Thought:

Let's represent the cost of the ball as x. 

Since the bat costs $1.00 more than the ball, we can represent the cost of the bat as x + $1.00.

The total cost of the bat and ball together is $1.10, so we can write it as:
x + (x + $1.00) = $1.10

Now, we can solve for x by combining like terms:
2x + $1.00 = $1.10
2x = $1.10 - $1.00
2x = $0.10
x = $0.10 / 2
x = $0.05

Therefore, the ball costs 5 cents.

In [21]:
# Zero-shot CoT: Just add 'Let's think step by step'
def zero_shot_cot(problem):
    prompt = f"{problem}\n\nLet's think step by step."
    response = client.chat.completions.create(
        model='gpt-3.5-turbo',
        messages=[{'role': 'user', 'content': prompt}],
        temperature=0.2
    )
    return response.choices[0].message.content

md("### 🧠 Zero-Shot CoT on Logic Problem")
md(zero_shot_cot(LOGIC_PROBLEM))

### 🧠 Zero-Shot CoT on Logic Problem

1. All roses are flowers: This means that every rose is a type of flower.

2. Some flowers fade quickly: This statement does not specify which flowers fade quickly, only that there are some flowers that do.

Based on these two statements, we cannot definitively conclude that some roses fade quickly. While all roses are flowers, it is possible that the roses in question do not fall into the category of flowers that fade quickly. The information provided does not directly link the fading characteristic to roses specifically.

## 3. Self-Consistency Sampling (~25 min)

Generate multiple solutions and take the majority answer.

In [22]:
def extract_number(text):
    '''Extract the final numerical answer.'''
    numbers = re.findall(r'\b(\d+)\b', text)
    return int(numbers[-1]) if numbers else None

def solve_with_cot_temp(problem, temperature=0.2):
    '''Solve problem using CoT with configurable temperature.'''
    prompt = f'''Solve this problem step by step.

Problem: {problem}

Let me work through this step by step:'''
    
    response = client.chat.completions.create(
        model='gpt-3.5-turbo',
        messages=[{'role': 'user', 'content': prompt}],
        temperature=temperature
    )
    return response.choices[0].message.content

def self_consistent_solve(problem, n_samples=5, temperature=0.2):
    '''Solve using self-consistency: generate multiple answers and vote.
    
    Uses higher temperature to get diverse reasoning paths.
    '''
    answers = []
    
    for i in range(n_samples):
        solution = solve_with_cot_temp(problem, temperature=temperature)
        answer = extract_number(solution)
        answers.append(answer)
        print(f'Sample {i+1}: Answer = {answer}')
    
    # Majority vote
    counter = Counter([a for a in answers if a is not None])
    if counter:
        final_answer = counter.most_common(1)[0][0]
        confidence = counter[final_answer] / n_samples
        return final_answer, confidence
    return None, 0

md("### 🎲 Self-Consistency (5 samples)")
answer, conf = self_consistent_solve(MATH_PROBLEM, n_samples=5, temperature=1)
md(f"\n---\n\n**Final Answer:** `{answer}` (confidence: **{conf:.0%}**)")

### 🎲 Self-Consistency (5 samples)

Sample 1: Answer = 11
Sample 2: Answer = 11
Sample 3: Answer = 11
Sample 4: Answer = 11
Sample 5: Answer = 11



---

**Final Answer:** `11` (confidence: **100%**)

In [23]:
# Now try Self-Consistency on a HARDER problem where models may disagree
md("### 🎲 Self-Consistency on TRICKY Problem (Language Students)")
md("> **Correct answer: 60 students**")
md(f"\n**Problem:** {TRICKY_PROBLEM}")

answer, conf = self_consistent_solve(TRICKY_PROBLEM, n_samples=5, temperature=1)
md(f"\n---\n\n**Final Answer:** `{answer}` (confidence: **{conf:.0%}**)")

if conf < 1.0:
    md("\n> ⚠️ **Notice:** The model gave different answers across samples! This shows why self-consistency is valuable - it reveals uncertainty.")

### 🎲 Self-Consistency on TRICKY Problem (Language Students)

> **Correct answer: 60 students**


**Problem:** In a school, 1/2 of the students study French, 1/3 study Spanish, 
and 1/6 study both languages. If 20 students study neither language, 
how many students are in the school?

Sample 1: Answer = 60
Sample 2: Answer = 60
Sample 3: Answer = 60
Sample 4: Answer = 60
Sample 5: Answer = 120



---

**Final Answer:** `60` (confidence: **80%**)


> ⚠️ **Notice:** The model gave different answers across samples! This shows why self-consistency is valuable - it reveals uncertainty.

## 4. Prompt Chaining (~25 min)

Break complex tasks into a sequence of simpler prompts.

In [ ]:
def prompt_chain(task):
    '''Execute a multi-step task using prompt chaining.'''
    
    # Step 1: Understand and break down the problem
    step1 = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[{'role': 'user', 'content': f'Identify the key components of this problem: {task}'}],
        temperature=0
    ).choices[0].message.content
    md("#### Step 1 - Problem Analysis")
    md(step1)
    
    # Step 2: Plan solution approach
    step2 = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[{'role': 'user', 'content': f'Based on this analysis: {step1}\n\nOutline the solution steps:'}],
        temperature=0
    ).choices[0].message.content
    md("---\n#### Step 2 - Solution Plan")
    md(step2)
    
    # Step 3: Execute and get final answer
    step3 = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[{'role': 'user', 'content': f'Follow this plan: {step2}\n\nExecute and give the final answer:'}],
        temperature=0
    ).choices[0].message.content
    md("---\n#### Step 3 - Final Answer")
    md(step3)
    return step3

COMPLEX_PROBLEM = '''A store sells apples for $2 each and oranges for $3 each.
John spends $24 total and buys 10 fruits. How many of each did he buy?'''

md("### 🔗 Prompt Chaining Example")
prompt_chain(COMPLEX_PROBLEM)

### 🔗 Prompt Chaining Example

#### Step 1 - Problem Analysis

To solve the problem, we need to identify the key components and set up a system of equations based on the information provided. Here are the key components:

1. **Variables**:
   - Let \( x \) be the number of apples John buys.
   - Let \( y \) be the number of oranges John buys.

2. **Cost of Fruits**:
   - Apples cost $2 each.
   - Oranges cost $3 each.

3. **Total Amount Spent**:
   - John spends a total of $24.

4. **Total Number of Fruits**:
   - John buys a total of 10 fruits.

5. **Equations**:
   - From the total number of fruits: 
     \[
     x + y = 10
     \]
   - From the total amount spent:
     \[
     2x + 3y = 24
     \]

With these components, we can solve the system of equations to find the values of \( x \) and \( y \).

---
#### Step 2 - Solution Plan

To solve the system of equations derived from the problem, we can follow these steps:

### Step 1: Write Down the Equations
We have the following two equations based on the problem statement:
1. \( x + y = 10 \)  (Equation 1: Total number of fruits)
2. \( 2x + 3y = 24 \)  (Equation 2: Total amount spent)

### Step 2: Solve One Equation for One Variable
We can solve Equation 1 for \( y \):
\[
y = 10 - x
\]

### Step 3: Substitute into the Second Equation
Next, we substitute the expression for \( y \) from Equation 1 into Equation 2:
\[
2x + 3(10 - x) = 24
\]

### Step 4: Simplify the Equation
Now, simplify the equation:
\[
2x + 30 - 3x = 24
\]
Combine like terms:
\[
-1x + 30 = 24
\]

### Step 5: Solve for \( x \)
Rearranging gives:
\[
-1x = 24 - 30
\]
\[
-1x = -6
\]
\[
x = 6
\]

### Step 6: Substitute Back to Find \( y \)
Now that we have \( x \), we can find \( y \) using the expression we derived in Step 2:
\[
y = 10 - x = 10 - 6 = 4
\]

### Step 7: Conclusion
We have found the values:
- \( x = 6 \) (John buys 6 apples)
- \( y = 4 \) (John buys 4 oranges)

### Step 8: Verification
To ensure our solution is correct, we can verify by checking both original equations:
1. Total number of fruits: \( 6 + 4 = 10 \) (Correct)
2. Total amount spent: \( 2(6) + 3(4) = 12 + 12 = 24 \) (Correct)

Thus, the solution is verified, and John buys 6 apples and 4 oranges.

---
#### Step 3 - Final Answer

The final answer is that John buys 6 apples and 4 oranges.

'The final answer is that John buys 6 apples and 4 oranges.'

## 🎉 Summary

- **Chain-of-Thought**: Break problems into explicit reasoning steps
- **Self-Consistency**: Multiple samples + majority voting for robustness
- **Prompt Chaining**: Sequential prompts for complex multi-step tasks

### Best Practices
- Use CoT for math and logic problems
- Use self-consistency when accuracy is critical
- Use chaining when tasks have clear phases